Creating Schema



In [0]:
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%python
import zipfile
zip_path = "/Volumes/workspace/bronze/raw_data_volume/ecommerce_dataset/olist_customers_dataset.csv.zip"
# with zipfile.ZipFile(zip_path, 'r') as z:
#     print(z.namelist())
display(dbutils.fs.ls("/Volumes/workspace/bronze/raw_data_volume/ecommerce_dataset"))

O1_Bronze_Ingestion -Imports


In [0]:
%python """ Imports"""
from pyspark.sql import functions as F
import zipfile
import os

In [0]:
%python  """Path"""
volume_path = "/Volumes/workspace/bronze/raw_data_volume/ecommerce_dataset"
print("Volume Path:", volume_path)

In [0]:
%python  """Extract Customer CSV"""
# zip_path = "/Volumes/workspace/bronze/raw_data_volume/ecommerce_dataset"
# os.makedirs(extract_dir, exist_ok=True)
# with zipfile.ZipFile(zip_path, "r") as z:
#  z.extractall(extract_dir)
# print("Extracted Successfully")

for f in dbutils.fs.ls("/Volumes/workspace/bronze/raw_data_volume/ecommerce_dataset"):
    print(f.name)

In [0]:
%python  """Verify Extracted File"""
print(globals().keys())

In [0]:
%python """Read CSV with spark"""
customers_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("/Volumes/workspace/bronze/raw_data_volume/ecommerce_dataset/olist_customers_dataset.csv")
)
customers_df.show(5)
customers_df.printSchema()

In [0]:
%python """create Bronze table"""
# Read source file


# Full Refresh (Overwrite entire table)
(
    customers_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("bronze.customers")
)

print("Customers Full Refresh Completed")

In [0]:
%python """Validate"""

#  Validate After Full-Refresh , Method -1

print("Source Count :", customers_df.count())
print("Target Count :", spark.table("bronze.customers").count())

In [0]:
%python """CREATING REUSABLE INGESTION FUNCTION FOR REMAINING TABLES """
# Initial Full Load 
# Landing -> Bronze 
def load_csv_to_bronze(file_name, table_name):
    path = f"/Volumes/workspace/bronze/raw_data_volume/ecommerce_dataset/{file_name}"
    df = (
        spark.read
             .option("header", True)
             .option("inferSchema", True)
             .csv(path)
    )
    row_count = df.count()
    (
        df.write
          .format("delta")
          .mode("overwrite")
          .saveAsTable(f"bronze.{table_name}")
    )
    print(f"Loaded bronze.{table_name}")
    print(f"Rows: {row_count}")

In [0]:
%python """Load data"""
load_csv_to_bronze(
    "olist_order_items_dataset.csv",
    "order_items"
)

load_csv_to_bronze(
    "olist_order_payments_dataset.csv",
    "payments"
)

load_csv_to_bronze(
    "olist_products_dataset.csv",
    "products"
)

load_csv_to_bronze(
    "product_category_name_translation.csv",
    "category_name"
)


In [0]:
%python """Validate Bronze tables"""
spark.sql("SHOW TABLES IN bronze").show(truncate=False)

In [0]:
%python
tables = [
    "customers",
    "orders",
    "order_items",
    "payments",
    "products",
    "reviews",
    "sellers"
]
for table in tables:
    cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM bronze.{table}").collect()[0]["cnt"]
    print(f"{table}: {cnt:,}")